# Phase 1 - Supervised Baseline Segmentation, full improved version (LG-PM project)

**Purpose.** Reference segmentation model for the whole study: a fully
supervised ResNet34-encoder U-Net trained on TN3K and evaluated (1) in-domain
on the official TN3K test set and (2) zero-shot cross-domain on TNUI-2021 and
DDTI. Phase 2 (support-conditioned prototype segmentation) is benchmarked
against this reference at matched annotation budgets.

**This file.** Complete standalone notebook - every section of the original
Phase 1 (manifests, Phase 0 verification, mask audit with visual threshold
figure, sanity grids, CPU latency) is retained, with the Phase 1b upgrades
layered in:

| Change | Plain Phase 1 | This notebook |
|---|---|---|
| Input size | 256 (batch 16) | **384 (batch 8)** |
| Encoder | ResNet18 | **ResNet34** (same stage widths) |
| Loss | BCE + Dice | **BCE + Tversky (0.3/0.7)** |
| Augmentation | flip/rotate/BC/noise/blur | **+ affine scale/translate + elastic** |
| Weights | last best | **EMA**, selection on EMA val Dice |
| Threshold | fixed 0.5 | **tuned on internal validation only, then frozen (Section 7)** |
| Inference | single view | **TTA** (identity/hflip/+-5 deg/gain), reported next to non-TTA |
| Ensemble | single seed | optional multi-seed via `SEEDS_TO_RUN` |

**Compliance.** The threshold sweep uses only the TN3K internal validation
split, runs before any test/external evaluation, and the chosen value is
frozen. TNUI-2021 and DDTI are never used for training, selection, or
thresholding.

**Runtime.** ~35-55 min/seed on a Kaggle T4. Set
`SEEDS_TO_RUN = [42, 7, 2024]` for a 3-seed ensemble (~2-2.5 h).

**Inputs.** Mounted datasets as audited in Phase 0
(`/kaggle/input/datasets/jonbon8397/...`). Manifests are rebuilt from
`/kaggle/input` with the Phase 0 rules (SHA-256 de-duplication, priority
train < val < test) - no dependency on earlier `/kaggle/working` files.

**Outputs** (`/kaggle/working/phase1_segmentation/`): manifests, optional
verification report, mask audit CSV + threshold figure, per-seed checkpoints
and histories, threshold sweep, per-case metrics (TTA and non-TTA),
aggregate metrics with bootstrap 95% CIs, qualitative panels, GPU+CPU
latency snapshot, run summary.

## 0. Configuration

In [ ]:
from pathlib import Path
import json, time, random, hashlib, warnings, copy
import numpy as np
import pandas as pd
from PIL import Image
import matplotlib.pyplot as plt
import cv2
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.transforms.functional as TF
from torch.utils.data import Dataset, DataLoader
from IPython.display import display

warnings.filterwarnings("ignore")

SEED = 42
def set_seed(seed=SEED):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
set_seed()

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
PIN = DEVICE.type == "cuda"

INPUT_ROOT = Path("/kaggle/input/datasets")
AUTHOR_ROOT = INPUT_ROOT / "jonbon8397"
PATHS = {
    "tn3k": AUTHOR_ROOT / "tn3k-data" / "Thyroid Dataset" / "tn3k",
    "ddti": AUTHOR_ROOT / "tn3k-data" / "Thyroid Dataset" / "DDTI dataset" / "DDTI",
    "tnui": AUTHOR_ROOT / "tnui-2021" / "TNUI-2021-" / "thyroid_data",
}
OUTPUT_ROOT = Path("/kaggle/working/phase1_segmentation")
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

# -------- hyperparameters (fixed before external evaluation) --------
IMG_SIZE     = 384
BATCH_SIZE   = 8
NUM_WORKERS  = 2
EPOCHS       = 60
LR           = 1e-3
WEIGHT_DECAY = 1e-4
PATIENCE     = 12
MASK_THRESH  = 127          # see Section 2 visual audit
VAL_FRACTION = 0.15
N_BOOT       = 2000
ENCODER      = "resnet34"   # resnet18 | resnet34 (same stage widths)
TVERSKY_A    = 0.3          # FP weight
TVERSKY_B    = 0.7          # FN weight
EMA_DECAY    = 0.999
SEEDS_TO_RUN = [SEED]       # e.g. [42, 7, 2024] for a 3-seed ensemble
THRS         = np.round(np.arange(0.30, 0.701, 0.02), 3)   # validation-only sweep
RUN_TTA      = True

print("torch:", torch.__version__, "| device:", DEVICE)
for k, v in PATHS.items():
    print(f"{k:6s} | {'OK' if v.exists() else 'MISSING'} | {v}")

## 1. Manifest reconstruction with leakage controls

Reproduces the Phase 0 cleaning inside this notebook:

1. pair images and masks by filename stem;
2. hash every image (SHA-256);
3. drop later copies of exact duplicates, keeping the higher-priority split
   (TN3K: trainval > test; TNUI: train > val > test);
4. split TN3K trainval into internal train/val (image-level - TN3K provides
   no patient identifiers; **recorded limitation**);
5. assert zero hash overlap between training and all evaluation pools.

In [ ]:
IMG_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff"}

def image_files(folder):
    folder = Path(folder)
    if not folder.exists():
        return []
    return sorted(p for p in folder.rglob("*") if p.is_file() and p.suffix.lower() in IMG_EXTS)

def sha256_file(path, chunk=1 << 20):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for blk in iter(lambda: f.read(chunk), b""):
            h.update(blk)
    return h.hexdigest()

def pair_images_masks(image_dir, mask_dir, dataset, split):
    imgs = {p.stem: p for p in image_files(image_dir)}
    msks = {p.stem: p for p in image_files(mask_dir)}
    common = sorted(set(imgs) & set(msks))
    df = pd.DataFrame([{"dataset": dataset, "split": split, "id": k,
                        "image": str(imgs[k]), "mask": str(msks[k])} for k in common])
    n_mm, n_mi = len(set(imgs) - set(msks)), len(set(msks) - set(imgs))
    if n_mm or n_mi:
        print(f"  WARNING {dataset}/{split}: missing_masks={n_mm} missing_images={n_mi}")
    return df

def add_hashes(df):
    df = df.copy()
    df["sha256"] = df["image"].apply(lambda p: sha256_file(p) if Path(p).exists() else None)
    return df

def drop_exact_duplicates(df, priority):
    df = df.copy()
    df["_p"] = df["split"].map(priority).fillna(99)
    df = df.sort_values(["_p", "id"], kind="stable")
    keep = ~df.duplicated("sha256", keep="first")
    kept = df[keep].drop(columns="_p").reset_index(drop=True)
    dropped = df[~keep].drop(columns="_p").reset_index(drop=True)
    return kept, dropped

# ---- TN3K (source) ----
tn3k = pd.concat([
    pair_images_masks(PATHS["tn3k"]/"trainval-image", PATHS["tn3k"]/"trainval-mask", "TN3K", "trainval"),
    pair_images_masks(PATHS["tn3k"]/"test-image",     PATHS["tn3k"]/"test-mask",     "TN3K", "test"),
], ignore_index=True)
tn3k = add_hashes(tn3k)
tn3k, tn3k_dropped = drop_exact_duplicates(tn3k, {"trainval": 0, "test": 1})

rng = np.random.RandomState(SEED)
tv = tn3k[tn3k.split == "trainval"].copy()
perm = rng.permutation(len(tv))
val_ids = set(tv.iloc[perm[: int(round(VAL_FRACTION * len(tv)))]].id)
tv["split"] = np.where(tv.id.isin(val_ids), "val", "train")
tn3k = pd.concat([tv, tn3k[tn3k.split == "test"]], ignore_index=True)

# ---- TNUI-2021 (external only) ----
tnui = pd.concat([
    pair_images_masks(PATHS["tnui"]/s/"images", PATHS["tnui"]/s/"masks", "TNUI-2021", s)
    for s in ["train", "val", "test"]
], ignore_index=True)
tnui = add_hashes(tnui)
tnui, tnui_dropped = drop_exact_duplicates(tnui, {"train": 0, "val": 1, "test": 2})

# ---- DDTI (external only) ----
ddti = add_hashes(pair_images_masks(
    PATHS["ddti"]/"1_or_data"/"image", PATHS["ddti"]/"1_or_data"/"mask", "DDTI", "external"))
ddti, ddti_dropped = drop_exact_duplicates(ddti, {"external": 0})

SETS = {"TN3K": tn3k, "TNUI-2021": tnui, "DDTI": ddti}
for name, df in SETS.items():
    df.to_csv(OUTPUT_ROOT / f"manifest_{name.lower().replace('-', '_')}.csv", index=False)
pd.concat([tn3k_dropped, tnui_dropped, ddti_dropped]).to_csv(
    OUTPUT_ROOT / "dropped_exact_duplicates.csv", index=False)

h = {s: set(tn3k[tn3k.split == s].sha256) for s in ["train", "val", "test"]}
assert not (h["train"] & h["val"]) and not (h["train"] & h["test"]) and not (h["val"] & h["test"]), \
    "hash overlap between TN3K internal splits"
assert not (h["train"] & set(tnui.sha256)), "train/external overlap TN3K-TNUI"
assert not (h["train"] & set(ddti.sha256)), "train/external overlap TN3K-DDTI"

summary = pd.DataFrame([
    {"dataset": n, "split": s, "n": int((df.split == s).sum())}
    for n, df in SETS.items() for s in sorted(df.split.unique())
])
display(summary)
print("Dropped exact duplicates: TN3K=%d, TNUI=%d, DDTI=%d"
      % (len(tn3k_dropped), len(tnui_dropped), len(ddti_dropped)))
print("Leakage assertions passed.")

### 1.1 Optional - verify against the Phase 0 audit manifests

If you upload the Phase 0 clean manifests (`tn3k_clean.csv`,
`tnui_2021_clean.csv`, `ddti_clean.csv`, `all_dropped_duplicates.csv`) as a
Kaggle dataset and attach it, this cell verifies that the manifests rebuilt
above are **identical** to the audited ones (by image hash, per split). If
the files are not mounted, the cell reports and skips - the training does
not depend on them.

In [ ]:
ref_files = {}
for pat in ["tn3k_clean.csv", "tnui_2021_clean.csv", "ddti_clean.csv",
            "all_dropped_duplicates.csv"]:
    hits = list(Path("/kaggle/input").rglob(pat))
    if hits:
        ref_files[pat] = hits[0]

if not ref_files:
    print("Phase 0 reference manifests not mounted - verification skipped (optional).")
    print("To enable: upload the *_clean.csv files as a Kaggle dataset and attach it.")
else:
    print("Found reference manifests:")
    for k, v in ref_files.items():
        print(" ", k, "->", v)
    checks = []
    rebuilt = {"tn3k_clean.csv": tn3k, "tnui_2021_clean.csv": tnui, "ddti_clean.csv": ddti}
    for fname, df_new in rebuilt.items():
        if fname not in ref_files:
            checks.append({"file": fname, "status": "reference missing"})
            continue
        ref = pd.read_csv(ref_files[fname])
        if "sha256" in ref.columns:
            ref_h, new_h = set(ref["sha256"]), set(df_new["sha256"])
            checks.append({"file": fname,
                           "status": "PASS (hash match)" if ref_h == new_h else "MISMATCH (hash)",
                           "ref_n": len(ref), "new_n": len(df_new),
                           "only_ref": len(ref_h - new_h), "only_new": len(new_h - ref_h)})
        else:
            ok = set(ref["id"]) == set(df_new["id"])
            checks.append({"file": fname, "status": "PASS (id match)" if ok else "MISMATCH (id)",
                           "ref_n": len(ref), "new_n": len(df_new)})
    if "all_dropped_duplicates.csv" in ref_files:
        ref_drop = set(pd.read_csv(ref_files["all_dropped_duplicates.csv"])["sha256"])
        new_drop = set(pd.concat([tn3k_dropped, tnui_dropped, ddti_dropped])["sha256"])
        checks.append({"file": "all_dropped_duplicates.csv",
                       "status": "PASS (hash match)" if ref_drop == new_drop else "MISMATCH (hash)",
                       "ref_n": len(ref_drop), "new_n": len(new_drop)})
    ver = pd.DataFrame(checks)
    display(ver)
    ver.to_csv(OUTPUT_ROOT / "manifest_verification.csv", index=False)
    bad = ver[ver.status.str.contains("MISMATCH")]
    if len(bad):
        print("MISMATCH found - inspect before training:", list(bad["file"]))
    else:
        print("All mounted reference manifests reproduced exactly (hash-level).")

## 2. Mask binarisation audit

Phase 0 found TN3K masks stored as JPEG with ~17-28 unique grey values
(compression artefacts). This cell compares the thresholds `>0`, `>=127`,
`>=250` on random TN3K cases. **Confirm visually that the green (>=127)
contour reproduces the lesion outline**; change `MASK_THRESH` only if the
overlays show the default is wrong (and record the change). TNUI/DDTI masks
are near-binary, so one shared threshold is safe.

In [ ]:
def read_gray(p):
    return np.asarray(Image.open(p).convert("L"))

audit_rows = []
for name, df in SETS.items():
    for _, r in df.sample(min(30, len(df)), random_state=SEED).iterrows():
        a = read_gray(r["mask"])
        audit_rows.append({"dataset": name, "id": r["id"], "unique": int(np.unique(a).size),
                           "min": int(a.min()), "max": int(a.max()),
                           "fg_gt0": float((a > 0).mean()),
                           "fg_ge127": float((a >= 127).mean()),
                           "fg_ge250": float((a >= 250).mean())})
audit_df = pd.DataFrame(audit_rows)
audit_df.to_csv(OUTPUT_ROOT / "mask_threshold_audit.csv", index=False)
display(audit_df.groupby("dataset")[["unique", "fg_gt0", "fg_ge127", "fg_ge250"]]
        .agg(["mean", "min", "max"]).round(4))

fig, axes = plt.subplots(2, 5, figsize=(18, 7))
for col, (_, r) in enumerate(tn3k[tn3k.split == "train"].sample(5, random_state=3).iterrows()):
    img = np.asarray(Image.open(r["image"]).convert("RGB"))
    a = read_gray(r["mask"])
    axes[0, col].imshow(img); axes[0, col].set_title(r["id"]); axes[0, col].axis("off")
    rgb = np.zeros((*a.shape, 3), float)
    rgb[..., 0] = (a > 0)      # red: permissive
    rgb[..., 1] = (a >= 127)   # green: default
    rgb[..., 2] = (a >= 250)   # blue: strict
    axes[1, col].imshow(rgb); axes[1, col].axis("off")
    axes[1, col].set_title("R:>0  G:>=127  B:>=250 (white=agree)")
plt.suptitle("TN3K mask-threshold comparison")
plt.tight_layout(); plt.show()
print("MASK_THRESH =", MASK_THRESH)

## 3. Preprocessing and augmentation

- **Letterbox** to 384x384 (aspect preserved, zero padding); masks use
  nearest interpolation and background padding.
- **Train-time augmentation** mirrors the ultrasound-specific list in the
  Methods: horizontal flip, +-10 deg rotation, brightness/contrast (gain),
  speckle-like noise, mild blur, **plus affine scale/translate and elastic
  deformation** (with graceful fallbacks for older `albumentations`).
  Identical geometric transforms are applied to image and mask. **Check the
  printed transform count** - if it fell back to flip-only, report it.
- 3-channel input with ImageNet normalisation.

In [ ]:
def letterbox(arr, size, is_mask):
    h, w = arr.shape[:2]
    s = size / max(h, w)
    nh, nw = max(1, int(round(h * s))), max(1, int(round(w * s)))
    interp = cv2.INTER_NEAREST if is_mask else cv2.INTER_LINEAR
    rz = cv2.resize(arr, (nw, nh), interpolation=interp)
    t, l = (size - nh) // 2, (size - nw) // 2
    if arr.ndim == 2:
        canvas = np.zeros((size, size), dtype=rz.dtype)
        canvas[t:t + nh, l:l + nw] = rz
        return (canvas > 0).astype(np.uint8) if is_mask else canvas
    canvas = np.zeros((size, size, arr.shape[2]), dtype=rz.dtype)
    canvas[t:t + nh, l:l + nw, :] = rz
    return canvas

try:
    import albumentations as A
    try:
        geo = A.Affine(scale=(0.85, 1.15), translate_percent=(0.05, 0.05),
                       rotate=(-10, 10), p=0.6)
    except AttributeError:
        geo = A.ShiftScaleRotate(shift_limit=0.05, scale_limit=0.15, rotate_limit=10, p=0.6)
    augs = [A.HorizontalFlip(p=0.5), geo,
            A.RandomBrightnessContrast(brightness_limit=0.25, contrast_limit=0.25, p=0.5),
            A.GaussNoise(p=0.25), A.GaussianBlur(blur_limit=3, p=0.15)]
    try:
        augs.append(A.ElasticTransform(alpha=30, sigma=5, p=0.25))
    except Exception as e:
        print("elastic skipped:", repr(e))
    train_tf = A.Compose(augs)
    print("albumentations:", A.__version__, "| n_transforms:", len(augs))
except Exception as e:
    train_tf = None
    print("albumentations unavailable - flip-only fallback:", repr(e))

MEAN = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
STD  = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)

class ThyroidSegDataset(Dataset):
    def __init__(self, df, train=False):
        self.df = df.reset_index(drop=True)
        self.train = train
    def __len__(self):
        return len(self.df)
    def __getitem__(self, i):
        r = self.df.iloc[i]
        img = read_gray(r["image"])
        msk = (read_gray(r["mask"]) >= MASK_THRESH).astype(np.uint8)
        img3 = cv2.cvtColor(img, cv2.COLOR_GRAY2RGB)
        if self.train and train_tf is not None:
            out = train_tf(image=img3, mask=msk)
            img3, msk = out["image"], (np.asarray(out["mask"]) > 0).astype(np.uint8)
        elif self.train and random.random() < 0.5:
            img3 = np.ascontiguousarray(img3[:, ::-1])
            msk = np.ascontiguousarray(msk[:, ::-1])
        img3 = letterbox(img3, IMG_SIZE, is_mask=False)
        msk = letterbox(msk, IMG_SIZE, is_mask=True)
        x = torch.from_numpy(img3.transpose(2, 0, 1)).float() / 255.0
        x = (x - MEAN) / STD
        return x, torch.from_numpy(msk[None]).float(), str(r["id"])

train_df = tn3k[tn3k.split == "train"].reset_index(drop=True)
val_df   = tn3k[tn3k.split == "val"].reset_index(drop=True)
test_df  = tn3k[tn3k.split == "test"].reset_index(drop=True)

def make_loader(df, train=False, seed=SEED):
    g = torch.Generator().manual_seed(seed)
    return DataLoader(ThyroidSegDataset(df, train), batch_size=BATCH_SIZE,
                      shuffle=train, num_workers=NUM_WORKERS, pin_memory=PIN,
                      drop_last=train, generator=g if train else None)

train_loader = make_loader(train_df, train=True)
val_loader   = make_loader(val_df)
print(f"TN3K internal: train={len(train_df)}  val={len(val_df)}  test={len(test_df)}")

### 3.1 Sanity check - augmented samples

Verify image/mask alignment after the full augmentation + letterbox
pipeline (including elastic). If outlines drift from the lesion, stop and
fix the transform code before training.

In [ ]:
x_b, y_b, ids_b = next(iter(train_loader))
inv = lambda x: (x * STD + MEAN).clamp(0, 1)
fig, axes = plt.subplots(4, 3, figsize=(9, 12))
for i in range(4):
    img = inv(x_b[i]).permute(1, 2, 0).numpy()
    m = y_b[i, 0].numpy()
    axes[i, 0].imshow(img); axes[i, 0].set_title(ids_b[i]); axes[i, 0].axis("off")
    axes[i, 1].imshow(m, cmap="gray"); axes[i, 1].axis("off")
    axes[i, 2].imshow(img); axes[i, 2].imshow(m, cmap="jet", alpha=0.35); axes[i, 2].axis("off")
plt.suptitle("Augmented training samples @384 (image / mask / overlay)")
plt.tight_layout(); plt.show()

## 4. Model - ResNet34-encoder U-Net + EMA

Manual implementation (no external segmentation package). If ImageNet
weights cannot be downloaded, it falls back to random initialisation -
**check the printed line**, since that materially changes reference
performance. Training tracks an **exponential moving average** of the
weights alongside the raw weights; the best checkpoint stores whichever
validates higher (recorded as `which`).

In [ ]:
def conv_block(cin, cout):
    return nn.Sequential(
        nn.Conv2d(cin, cout, 3, padding=1, bias=False), nn.BatchNorm2d(cout), nn.ReLU(inplace=True),
        nn.Conv2d(cout, cout, 3, padding=1, bias=False), nn.BatchNorm2d(cout), nn.ReLU(inplace=True))

class Up(nn.Module):
    def __init__(self, cin, cskip, cout):
        super().__init__()
        self.conv = conv_block(cin + cskip, cout)
    def forward(self, x, skip):
        x = F.interpolate(x, size=skip.shape[-2:], mode="bilinear", align_corners=False)
        return self.conv(torch.cat([x, skip], dim=1))

_WMAP = {"resnet18": "ResNet18_Weights", "resnet34": "ResNet34_Weights"}

def build_model(pretrained=True):
    import torchvision.models as tvm
    try:
        if not pretrained:
            raise RuntimeError("pretrained=False")
        w = getattr(tvm, _WMAP[ENCODER]).IMAGENET1K_V1
        bb = getattr(tvm, ENCODER)(weights=w)
        print(f"ImageNet-pretrained {ENCODER} loaded.")
    except Exception as e:
        print("Pretrained weights UNAVAILABLE - random init:", repr(e))
        bb = getattr(tvm, ENCODER)(weights=None)
    class M(nn.Module):
        def __init__(self):
            super().__init__()
            self.stem = nn.Sequential(bb.conv1, bb.bn1, bb.relu)
            self.pool = bb.maxpool
            self.enc1, self.enc2, self.enc3, self.enc4 = bb.layer1, bb.layer2, bb.layer3, bb.layer4
            self.up4, self.up3 = Up(512, 256, 256), Up(256, 128, 128)
            self.up2, self.up1 = Up(128, 64, 64), Up(64, 64, 64)
            self.head = nn.Sequential(
                nn.Upsample(scale_factor=2, mode="bilinear", align_corners=False),
                conv_block(64, 32), nn.Conv2d(32, 1, 1))
        def forward(self, x):
            e0 = self.stem(x)
            e1 = self.enc1(self.pool(e0)); e2 = self.enc2(e1)
            e3 = self.enc3(e2); e4 = self.enc4(e3)
            d = self.up1(self.up2(self.up3(self.up4(e4, e3), e2), e1), e0)
            return self.head(d)
    return M()

class EMA:
    def __init__(self, model, decay=EMA_DECAY):
        self.decay = decay
        self.shadow = {k: (v.detach().clone().float() if v.dtype.is_floating_point
                           else v.detach().clone())
                       for k, v in model.state_dict().items()}
    @torch.no_grad()
    def update(self, model):
        for k, v in model.state_dict().items():
            s = self.shadow[k]
            if v.dtype.is_floating_point:
                s.mul_(self.decay).add_(v.detach().float(), alpha=1 - self.decay)
            else:
                s.copy_(v)

_m = build_model().to(DEVICE)
n_params = sum(p.numel() for p in _m.parameters())
out_shape = _m(torch.zeros(1, 3, IMG_SIZE, IMG_SIZE, device=DEVICE)).shape
print(f"parameters: {n_params/1e6:.2f} M | output shape: {tuple(out_shape)}")
del _m; torch.cuda.empty_cache()

## 5. Loss and metrics

BCE + **Tversky** with alpha=0.3 (FP) / beta=0.7 (FN): missed lesion pixels
cost more than spillover - appropriate for small, fuzzy-boundary nodules.
Per-case metrics: Dice, IoU, sensitivity, precision, HD95 (nan-safe).

In [ ]:
from scipy.ndimage import distance_transform_edt, binary_erosion

def tversky_loss(logits, targets, a=TVERSKY_A, b=TVERSKY_B, eps=1e-6):
    p = torch.sigmoid(logits)
    tp = (p * targets).sum(dim=(1, 2, 3))
    fp = (p * (1 - targets)).sum(dim=(1, 2, 3))
    fn = ((1 - p) * targets).sum(dim=(1, 2, 3))
    return (1 - (tp + eps) / (tp + a * fp + b * fn + eps)).mean()

def seg_loss(logits, targets):
    return F.binary_cross_entropy_with_logits(logits, targets) + tversky_loss(logits, targets)

def batch_dice(logits, y, thr=0.5):
    pred = (torch.sigmoid(logits) >= thr).float()
    num = 2 * (pred * y).sum((1, 2, 3))
    den = pred.sum((1, 2, 3)) + y.sum((1, 2, 3)) + 1e-6
    return num / den

def case_metrics(pred, gt):
    pred, gt = pred.astype(bool), gt.astype(bool)
    tp = int((pred & gt).sum()); fp = int((pred & ~gt).sum()); fn = int((~pred & gt).sum())
    empty = not (gt.any() or pred.any())
    dice = (2*tp)/(2*tp+fp+fn) if (2*tp+fp+fn) else (1.0 if empty else 0.0)
    iou  = tp/(tp+fp+fn) if (tp+fp+fn) else (1.0 if empty else 0.0)
    sens = tp/(tp+fn) if (tp+fn) else np.nan
    prec = tp/(tp+fp) if (tp+fp) else np.nan
    return dice, iou, sens, prec

def hd95(pred, gt):
    pred, gt = pred.astype(bool), gt.astype(bool)
    if not pred.any() or not gt.any():
        return np.nan
    pb = pred ^ binary_erosion(pred, border_value=0)
    gb = gt ^ binary_erosion(gt, border_value=0)
    if not pb.any() or not gb.any():
        return np.nan
    d = np.concatenate([distance_transform_edt(~gb)[pb], distance_transform_edt(~pb)[gb]])
    return float(np.percentile(d, 95))

## 6. Training (per seed, with EMA selection)

AdamW + ReduceLROnPlateau on the higher of raw/EMA val Dice, AMP on GPU,
early stopping (patience 12), best checkpoint per seed saved. Each epoch
prints raw and EMA validation Dice.

In [ ]:
def run_epoch(model, loader, opt=None, scaler=None, ema=None):
    train = opt is not None
    model.train() if train else model.eval()
    tl, n, dices = 0.0, 0, []
    ctx = torch.enable_grad() if train else torch.no_grad()
    with ctx:
        for x, y, _ in loader:
            x = x.to(DEVICE, non_blocking=PIN); y = y.to(DEVICE, non_blocking=PIN)
            with torch.cuda.amp.autocast(enabled=(DEVICE.type == "cuda")):
                logits = model(x)
                loss = seg_loss(logits, y)
            if train:
                opt.zero_grad(set_to_none=True)
                scaler.scale(loss).backward()
                scaler.step(opt); scaler.update()
                if ema is not None:
                    ema.update(model)
            bs = x.size(0); tl += loss.item() * bs; n += bs
            dices.extend(batch_dice(logits.detach(), y).cpu().numpy())
    return tl / max(n, 1), float(np.mean(dices))

def eval_with_ema(model, ema, loader):
    backup = {k: v.detach().clone() for k, v in model.state_dict().items()}
    model.load_state_dict(ema.shadow, strict=True)
    res = run_epoch(model, loader, opt=None)
    model.load_state_dict(backup, strict=True)
    return res

def train_one(seed):
    set_seed(seed)
    tr_loader = make_loader(train_df, train=True, seed=seed)
    va_loader = make_loader(val_df, seed=seed)
    model = build_model().to(DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    sched = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, mode="max", factor=0.5, patience=4)
    scaler = torch.cuda.amp.GradScaler(enabled=(DEVICE.type == "cuda"))
    ema = EMA(model)
    ckpt_path = OUTPUT_ROOT / f"best_{ENCODER}_seed{seed}.pt"
    best, best_ep, wait, t0 = -1.0, -1, 0, time.time()
    hist = []
    for ep in range(1, EPOCHS + 1):
        tr_l, tr_d = run_epoch(model, tr_loader, opt, scaler, ema)
        va_l, va_d = run_epoch(model, va_loader)
        _, va_de = eval_with_ema(model, ema, va_loader)
        sched.step(max(va_d, va_de))
        cur, which = (va_de, "ema") if va_de >= va_d else (va_d, "raw")
        improved = cur > best + 1e-4
        if improved:
            best, best_ep, wait = cur, ep, 0
            state = ema.shadow if which == "ema" else model.state_dict()
            torch.save({"model": {k: v.clone() for k, v in state.items()}, "which": which,
                        "epoch": ep, "val_dice": cur, "seed": seed, "encoder": ENCODER,
                        "config": {"img_size": IMG_SIZE, "mask_thresh": MASK_THRESH}}, ckpt_path)
        else:
            wait += 1
        hist.append({"seed": seed, "epoch": ep, "train_loss": tr_l, "train_dice": tr_d,
                     "val_loss": va_l, "val_dice": va_d, "val_dice_ema": va_de,
                     "lr": opt.param_groups[0]["lr"]})
        print(f"s{seed} ep {ep:02d} | tr loss {tr_l:.4f} dice {tr_d:.4f} | "
              f"val dice {va_d:.4f} ema {va_de:.4f}" + ("  *" if improved else ""))
        if wait >= PATIENCE:
            print("early stopping at epoch", ep)
            break
    print(f"seed {seed}: best val Dice {best:.4f} @ ep {best_ep} | {(time.time()-t0)/60:.1f} min")
    h = pd.DataFrame(hist)
    h.to_csv(OUTPUT_ROOT / f"history_seed{seed}.csv", index=False)
    del model, opt, scaler, ema
    torch.cuda.empty_cache()
    return ckpt_path, best, h

ckpts, bests, hists = [], [], []
for sd in SEEDS_TO_RUN:
    p, b, hsd = train_one(sd)
    ckpts.append(p); bests.append(b); hists.append(hsd)
hist_all = pd.concat(hists, ignore_index=True)
print("seeds:", SEEDS_TO_RUN, "| best val Dice per seed:", [round(b, 4) for b in bests])

### 6.1 Training curves (raw vs EMA validation Dice)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
for sd, hsd in zip(SEEDS_TO_RUN, hists):
    ax.plot(hsd.epoch, hsd.val_dice, alpha=0.4, label=f"s{sd} raw")
    ax.plot(hsd.epoch, hsd.val_dice_ema, label=f"s{sd} EMA")
ax.set_title("Validation Dice"); ax.set_xlabel("epoch"); ax.legend(ncol=2)
plt.tight_layout(); plt.show()

models = []
for p in ckpts:
    mck = build_model(pretrained=False).to(DEVICE)
    ck = torch.load(p, map_location=DEVICE)
    mck.load_state_dict(ck["model"]); mck.eval()
    models.append(mck)
    print(f"loaded {p.name} (which={ck['which']}, val Dice {ck['val_dice']:.4f})")

## 7. Threshold tuning - internal validation split ONLY

Sweeps sigmoid thresholds on the TN3K internal validation split using
seed-averaged probabilities (non-TTA). The selected threshold is **frozen**
before any test or external evaluation. Compliance: this cell must run
before Section 8, and the chosen value is recorded in the run summary.

In [ ]:
@torch.no_grad()
def ens_prob(x, use_tta=False):
    def views(z):
        yield z, (lambda q: q)
        if use_tta:
            yield torch.flip(z, [3]), (lambda q: torch.flip(q, [3]))
            yield TF.rotate(z, 5),  (lambda q: TF.rotate(q, -5))
            yield TF.rotate(z, -5), (lambda q: TF.rotate(q, 5))
            yield z * 1.05,         (lambda q: q)
    acc = 0
    for m in models:
        ps = 0
        nv = 1
        for z, invf in views(z=x):
            with torch.cuda.amp.autocast(enabled=(DEVICE.type == "cuda")):
                ps = ps + invf(torch.sigmoid(m(z)))
            nv += 1
        acc = acc + ps / nv
    return acc / len(models)

@torch.no_grad()
def tune_threshold(loader):
    TPs, FPs, FNs = [], [], []
    for x, y, _ in loader:
        yg = y.numpy().astype(bool)[:, 0]
        prob = ens_prob(x.to(DEVICE, non_blocking=PIN), use_tta=False).float().cpu().numpy()[:, 0]
        pb = prob[:, None] >= THRS[None, :, None, None]
        gt = yg[:, None]
        TPs.append((pb & gt).sum((2, 3))); FPs.append((pb & ~gt).sum((2, 3)))
        FNs.append((~pb & gt).sum((2, 3)))
    tp, fp, fn = np.concatenate(TPs), np.concatenate(FPs), np.concatenate(FNs)
    dice = 2 * tp / (2 * tp + fp + fn + 1e-6)
    per_thr = dice.mean(axis=0)
    sweep = pd.DataFrame({"thr": THRS, "val_dice": per_thr})
    sweep.to_csv(OUTPUT_ROOT / "threshold_sweep_val_only.csv", index=False)
    return float(THRS[int(np.argmax(per_thr))]), sweep

PRED_THRESH, sweep_df = tune_threshold(val_loader)
print(f"Tuned threshold (validation only): {PRED_THRESH}")
display(sweep_df.sort_values("val_dice", ascending=False).head(5).round(4))
plt.plot(sweep_df.thr, sweep_df.val_dice, marker="o")
plt.axvline(0.5, ls="--", c="grey", label="0.5 (fixed, plain Phase 1)")
plt.axvline(PRED_THRESH, ls="--", c="red", label=f"tuned {PRED_THRESH}")
plt.xlabel("threshold"); plt.ylabel("val Dice"); plt.legend(); plt.show()

## 8. Evaluation - in-domain and zero-shot cross-domain

Six evaluation pools (TN3K test; TNUI train/val/test/all; DDTI), each with
and without TTA, at the frozen threshold. TNUI-2021 and DDTI are fully
external: the model has never seen them and no selection step uses them.
Bootstrap 95% CIs are image-level (no patient linkage in these manifests -
**recorded limitation** to revisit for ThyUS2Path).

In [ ]:
@torch.no_grad()
def infer_case_metrics(df, set_name, use_tta=False):
    loader = make_loader(df.reset_index(drop=True), seed=SEED)
    rows = []
    for x, y, ids in loader:
        prob = ens_prob(x.to(DEVICE, non_blocking=PIN), use_tta=use_tta).float().cpu().numpy()
        pred = (prob >= PRED_THRESH).astype(bool)
        gt = y.numpy().astype(bool)
        for i in range(len(ids)):
            d, j, s, p = case_metrics(pred[i, 0], gt[i, 0])
            rows.append({"set": set_name, "tta": use_tta, "id": ids[i], "dice": d,
                         "iou": j, "sens": s, "prec": p, "hd95": hd95(pred[i, 0], gt[i, 0])})
    return pd.DataFrame(rows)

def boot_ci(v, n=N_BOOT):
    v = np.asarray(v, float); v = v[~np.isnan(v)]
    if len(v) == 0:
        return np.nan, np.nan
    rng = np.random.RandomState(SEED)
    s = np.array([np.mean(rng.choice(v, len(v), replace=True)) for _ in range(n)])
    return float(np.percentile(s, 2.5)), float(np.percentile(s, 97.5))

eval_jobs = [
    ("TN3K test (in-domain)",  test_df),
    ("TNUI-2021 train (ext)",  tnui[tnui.split == "train"]),
    ("TNUI-2021 val (ext)",    tnui[tnui.split == "val"]),
    ("TNUI-2021 test (ext)",   tnui[tnui.split == "test"]),
    ("TNUI-2021 all (ext)",    tnui),
    ("DDTI (ext)",             ddti),
]
frames = []
for name, df in eval_jobs:
    frames.append(infer_case_metrics(df, name, use_tta=False))
    if RUN_TTA:
        frames.append(infer_case_metrics(df, name, use_tta=True))
per_case = pd.concat(frames, ignore_index=True)
per_case.to_csv(OUTPUT_ROOT / "per_case_metrics.csv", index=False)

agg_rows = []
for (name, tta), grp in per_case.groupby(["set", "tta"], sort=False):
    for metric in ["dice", "iou", "sens", "prec"]:
        lo, hi = boot_ci(grp[metric])
        agg_rows.append({"set": name, "tta": tta, "metric": metric,
                         "mean": grp[metric].mean(), "sd": grp[metric].std(),
                         "ci95_lo": lo, "ci95_hi": hi, "n": len(grp)})
    agg_rows.append({"set": name, "tta": tta, "metric": "hd95_median",
                     "mean": float(np.nanmedian(grp["hd95"])), "sd": np.nan,
                     "ci95_lo": np.nan, "ci95_hi": np.nan,
                     "n": int(grp["hd95"].notna().sum())})
agg = pd.DataFrame(agg_rows)
agg.to_csv(OUTPUT_ROOT / "aggregate_metrics.csv", index=False)
disp = agg.pivot_table(index=["set", "tta"], columns="metric", values="mean").round(4)
display(disp)
for (name, tta), grp in per_case.groupby(["set", "tta"], sort=False):
    lo, hi = boot_ci(grp["dice"])
    print(f"{name:24s} tta={str(tta):5s}  Dice {grp['dice'].mean():.4f} ({lo:.4f}-{hi:.4f}) n={len(grp)}")

### 8.1 Qualitative review - best / median / worst

For TN3K test, TNUI-2021 test, and DDTI: three best, three median, three
worst cases by Dice (non-TTA seed-averaged probabilities). **Review the
worst rows** - they define the failure modes Phase 2 support conditioning
must address.

In [ ]:
def show_cases(df, pc, set_name):
    d = df.reset_index(drop=True)
    id2idx = {r["id"]: i for i, r in d.iterrows()}
    ds = ThyroidSegDataset(d, train=False)
    dices = pc["dice"].to_numpy()
    order = np.argsort(dices)
    picks = {"best": order[::-1][:3],
             "median": np.argsort(np.abs(dices - np.median(dices)))[:3],
             "worst": order[:3]}
    fig, axes = plt.subplots(9, 4, figsize=(12, 24))
    rr = 0
    for grp_name, idxs in picks.items():
        for j in idxs:
            cid = pc.iloc[j]["id"]
            x, y, _ = ds[id2idx[cid]]
            prob = ens_prob(x.unsqueeze(0).to(DEVICE), use_tta=False)[0, 0].float().cpu().numpy()
            pred = prob >= PRED_THRESH
            img = ((x * STD + MEAN).clamp(0, 1)).permute(1, 2, 0).numpy()
            gt = y[0].numpy()
            cols = [img, gt, pred, img]
            titles = [f"{grp_name} | {cid} | Dice {dices[j]:.3f}", "GT", "pred", "overlay"]
            for c in range(4):
                ax = axes[rr, c]
                if c == 3:
                    ax.imshow(img); ax.imshow(pred, cmap="jet", alpha=0.35)
                elif cols[c].ndim == 2:
                    ax.imshow(cols[c], cmap="gray")
                else:
                    ax.imshow(cols[c])
                ax.set_title(titles[c], fontsize=9); ax.axis("off")
            rr += 1
    fig.suptitle(set_name, fontsize=14)
    plt.tight_layout(); plt.show()

pc0 = per_case[~per_case.tta] if RUN_TTA else per_case
for set_name, df in [("TN3K test (in-domain)", test_df),
                     ("TNUI-2021 test (ext)", tnui[tnui.split == "test"]),
                     ("DDTI (ext)", ddti)]:
    show_cases(df, pc0[pc0.set == set_name].reset_index(drop=True), set_name)

## 9. Efficiency snapshot (GPU and CPU)

Teacher-track reference numbers for the Phase 7 edge-feasibility comparison:
parameter count, checkpoint size, warm inference latency on the training
device (single view and with TTA), and **CPU-only single-view latency** for
one seed's model.

In [ ]:
size_mb = ckpts[0].stat().st_size / 2**20
x1 = torch.zeros(1, 3, IMG_SIZE, IMG_SIZE, device=DEVICE)
def timed(fn, n=30, warm=5):
    for _ in range(warm):
        fn()
    if DEVICE.type == "cuda":
        torch.cuda.synchronize()
    t = time.perf_counter()
    for _ in range(n):
        fn()
    if DEVICE.type == "cuda":
        torch.cuda.synchronize()
    return (time.perf_counter() - t) / n * 1000.0

with torch.no_grad():
    lat_dev = timed(lambda: ens_prob(x1, use_tta=False))
    lat_dev_tta = timed(lambda: ens_prob(x1, use_tta=True), n=10, warm=2)

cpu_model = build_model(pretrained=False)
cpu_model.load_state_dict(torch.load(ckpts[0], map_location="cpu")["model"])
cpu_model.eval()
x_cpu = torch.zeros(1, 3, IMG_SIZE, IMG_SIZE)
with torch.no_grad():
    lat_cpu = timed(lambda: cpu_model(x_cpu), n=20, warm=3)
del cpu_model

eff = {"params_m_per_model": n_params / 1e6, "ckpt_mb_per_model": size_mb,
       "n_models": len(models), f"latency_ms_{DEVICE.type}_single": lat_dev,
       f"latency_ms_{DEVICE.type}_tta": lat_dev_tta, "latency_ms_cpu_single": lat_cpu}
print(json.dumps(eff, indent=2))
with open(OUTPUT_ROOT / "efficiency_snapshot.json", "w") as f:
    json.dump(eff, f, indent=2)

## 10. Run summary and artifacts

In [ ]:
run = {
    "phase": "1 - supervised baseline segmentation (full improved version)",
    "seeds": SEEDS_TO_RUN, "device": str(DEVICE), "torch": torch.__version__,
    "config": {"img_size": IMG_SIZE, "batch_size": BATCH_SIZE, "epochs_max": EPOCHS,
               "lr": LR, "wd": WEIGHT_DECAY, "patience": PATIENCE, "encoder": ENCODER,
               "tversky": [TVERSKY_A, TVERSKY_B], "ema_decay": EMA_DECAY,
               "mask_thresh": MASK_THRESH, "val_fraction": VAL_FRACTION,
               "run_tta": RUN_TTA},
    "tuned_threshold_val_only": PRED_THRESH,
    "best_val_dice_per_seed": dict(zip(map(str, SEEDS_TO_RUN), bests)),
    "aggregate": agg.astype(str).to_dict(orient="records"),
    "limitations": ["TN3K patient identifiers unavailable -> image-level split",
                    "bootstrap CIs are image-level",
                    "threshold tuned on internal val split only, frozen before test/external"],
}
with open(OUTPUT_ROOT / "run_summary.json", "w") as f:
    json.dump(run, f, indent=2)
print("Artifacts in", OUTPUT_ROOT)
for p in sorted(OUTPUT_ROOT.iterdir()):
    print(f"  {p.name:38s} {p.stat().st_size/1024:9.1f} KB")

## 11. Decision log and hand-off

**Confirmed by design:** TN3K source / TNUI-2021 + DDTI external; threshold
tuned on internal validation only and frozen; manifests identical rules to
Phase 0 (auto-verified in Section 1.1 when the Phase 0 CSVs are mounted).

**Fill in after the run:**
- [ ] pretrained encoder loaded? (Section 4 print)
- [ ] albumentations active? transform count? (Section 3 print)
- [ ] mask threshold >=127 visually confirmed? (Section 2 figure)

**Send back, one by one:**
1. best val Dice per seed + epochs run (Section 6);
2. tuned threshold + sweep curve note (Section 7);
3. the Section 8 pivot table - non-TTA **and** TTA rows;
4. the plain Phase 1 aggregate table if still available
   (`/kaggle/working/phase1_segmentation/aggregate_metrics.csv` from the
   earlier 256px run) - it stays in the paper as the plain-baseline row;
5. worst-case grids (TN3K test, TNUI test, DDTI);
6. efficiency numbers (GPU + CPU latency).

**Next:** Phase 2 - support-conditioned prototype segmentation (multi-scale
foreground/background masked-average prototypes on this encoder, nested
K in {1,5,10,20} supports, zero-shot + K-shot cross-domain evaluation on
TNUI-2021), benchmarked against this notebook's rows.